In [2]:
import pandas as pd
# Download: https://heidata.uni-heidelberg.de/dataset.xhtml?persistentId=doi:10.11588/data/0B5VML

# File paths
train_file = "germeval2018.training.txt"
test_file = "germeval2018.test.txt"

# Function to load and preprocess dataset
def load_germ_eval(file_path):
    # Read tab-separated file
    df = pd.read_csv(
        file_path,
        sep="\t",
        header=None,
        names=["text", "label", "sub_label"],
        encoding="utf-8"
    )

    # Convert labels:
    # OFFENSE -> 1
    # OTHER   -> 0
    df["label"] = df["label"].apply(lambda x: 1 if x == "OFFENSE" else 0)

    # Keep only text and label columns
    df = df[["text", "label"]]

    return df

# Load datasets
train_df = load_germ_eval(train_file)
test_df = load_germ_eval(test_file)

# Merge them
merged_df = pd.concat([train_df, test_df], ignore_index=True)

# Save final dataset
merged_df.to_csv("germeval2018.csv", index=False)

print(merged_df.head())
print("Total samples:", len(merged_df))

                                                text  label
0  @corinnamilborn Liebe Corinna, wir würden dich...      0
1  @Martin28a Sie haben ja auch Recht. Unser Twee...      0
2  @ahrens_theo fröhlicher gruß aus der schönsten...      0
3  @dushanwegner Amis hätten alles und jeden gewä...      0
4  @spdde kein verläßlicher Verhandlungspartner. ...      1
Total samples: 8407


In [3]:
import pandas as pd
# Downlaod: https://github.com/jagol/gahd/tree/main

# URLs
url_main = "https://raw.githubusercontent.com/jagol/gahd/refs/heads/main/gahd.csv"
url_dis = "https://raw.githubusercontent.com/jagol/gahd/refs/heads/main/gahd_disaggregated.csv"

# Load datasets
df_main = pd.read_csv(url_main)
df_dis = pd.read_csv(url_dis)

# Keep only required columns
df_main = df_main[["text", "label"]]
df_dis = df_dis[["text", "label"]]

# Merge both datasets
df = pd.concat([df_main, df_dis], ignore_index=True)

# Optional: drop duplicates (safe)
df = df.drop_duplicates()

# Save final CSV
df.to_csv("GAHD.csv", index=False)

print(df.head())
print("Total samples:", len(df))

                                                text  label
0  Ich denke, alles was man tun kann, ist, es zu ...      0
1  Es sind beides unsere Hunde, aber den schwarze...      0
2                                    Ich liebe Inder      0
3         Und dazu kommt auch noch die »Mode« LGBTQ.      1
4    Jeder weiß, dass Chinesen kleine Schwänze haben      1
Total samples: 10996


In [4]:
import pandas as pd

#Download: https://hasocfire.github.io/hasoc/2019/dataset.html

# Load TSV files (HASOC is tab-separated)
train_file = "german_dataset.tsv"
test_file  = "hasoc_de_test_gold.tsv"

def load_hasoc(path):
    df = pd.read_csv(path, sep="\t")

    # Ensure expected columns exist
    df = df[["text", "task_1"]]

    # Convert labels:
    # HOF = hate/offensive → 1
    # NOT = non-hate → 0
    df["label"] = df["task_1"].apply(lambda x: 1 if x == "HOF" else 0)

    return df[["text", "label"]]

# Load both splits
train_df = load_hasoc(train_file)
test_df  = load_hasoc(test_file)

# Merge
merged_df = pd.concat([train_df, test_df], ignore_index=True)

# Save
merged_df.to_csv("hasoc_german_merged.csv", index=False)

print(merged_df.head())
print("Total samples:", len(merged_df))

                                                text  label
0  Frank Rennicke – Ich bin stolz https://t.co/Cm...      0
1  ANSEHEN.....und danach bitte TEILEN...TEILEN.....      0
2  #Koeln Mohamed erkennt kein deutsches Recht so...      0
3  #SaudiArabien ist eine brutale islamische Dikt...      0
4  Bundespolizei #München hat im 1. Quartal 2019 ...      0
Total samples: 4669


In [5]:
!pip install emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 46.2 MB/s eta 0:00:00


In [7]:
import pandas as pd
import re
import emoji

def clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # remove URLs
    text = re.sub(r"http\S+|www\.\S+", "", text)

    # remove @mentions
    text = re.sub(r"@\w+", "", text)

    # remove ALL emojis using emoji library
    text = emoji.replace_emoji(text, replace="")

    # remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


def preprocess_df(df):
    df["text"] = df["text"].apply(clean_text)
    return df


germeval = pd.read_csv("germeval2018.csv")
hasoc    = pd.read_csv("hasoc_german_merged.csv")
gahd     = pd.read_csv("GAHD.csv")

germeval = preprocess_df(germeval)
hasoc    = preprocess_df(hasoc)
gahd     = preprocess_df(gahd)

germeval.to_csv("germeval_clean.csv", index=False)
hasoc.to_csv("hasoc_clean.csv", index=False)
gahd.to_csv("gahd_clean.csv", index=False)

print("Done: emojis + links + mentions removed")

Done: emojis + links + mentions removed
